# Setup

In [ ]:
# Import / install relevant packages
import requests
from requests_oauthlib import OAuth2Session
import xml.etree.ElementTree as et
import pprint
import json
import pandas as pd

from IPython.display import display
pd.set_option('display.max_columns', None)

!pip install xmltodict
import xmltodict

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

!pip install yahoo_oauth
from yahoo_oauth import OAuth2

from datetime import datetime, timedelta

# Authentication

In [ ]:
from yahoo_oauth import OAuth2

# Set up all yahoo the authentication stuff
# Testing the use of yahoo_oauth instead. Seems to be much
creds = {'consumer_key': 'dj0yJmk9TUwyZ3paYVVYTGFxJmQ9WVdrOWMxaDZNVFZOYzBJbWNHbzlNQT09JnM9Y29uc3VtZXJzZWNyZXQmc3Y9MCZ4PWE2', 'consumer_secret': '9923643fd2340d71052a5e900ec349de5d71d482'}
with open('oauth2.json', "w") as f:
   f.write(json.dumps(creds))

def get_oauth():
    """
    Retrieve a valid OAuth2 object.

    This function checks if the current token is valid and refreshes it if necessary.

    Returns:
        OAuth2: An instance of the OAuth2 class, ready to be used for authenticated requests.

    Raises:
        FileSystemError: If the 'oauth2.json' file is missing or cannot be read.
        RefreshTokenError: If the refresh token is invalid or expired
    """
    oauth = OAuth2(None, None, from_file='oauth2.json')
    if not oauth.token_is_valid():
        oauth.refresh_access_token()
    return oauth

def check_and_authenticate_user():
    """
    Verify the current user's authentication status and re-authenticate if necessary.

    This function attempts to retrieve a valid OAuth session. If the session is not valid,
    it handles exceptions by printing the error and could be extended to redirect to the
    login flow or raise an exception for further handling.

    Returns:
        None
    """
    try:
        oauth = get_oauth()
        # If we get here, we have a valid session.

    except Exception as e:
        # Handle exceptions and potentially re-authenticate.
        print("An error occurred: ", e)
        # Redirect to login flow or raise an exception.

# Game, League, Team, and Matchup Selection

In [ ]:
# Create function that auto extracts into dict format
def request_to_dict(request_url, print_output=False):
  """
  Make an authenticated request to a given URL, parse the XML response, convert it to JSON, and then to a dictionary.

  This function utilizes OAuth2 for authentication and xmltodict for parsing the XML response.

  Parameters:
  - request_url (str): The URL to which the OAuth2 authenticated request is to be sent.
  - print_output (bool, optional): If set to True, the JSON string is printed. Defaults to False.

  Returns:
  - dict: A dictionary representation of the JSON response obtained from the request URL.

  Raises:
  - requests.exceptions.RequestException: An error occurred while making the HTTP request.
  - xmltodict.expat.ExpatError: An error occurred while parsing the XML response to JSON.
  - json.JSONDecodeError: An error occurred during JSON decoding.
  """
  oauth = get_oauth()
  raw_json = json.dumps(xmltodict.parse(oauth.session.get(request_url).content))

  if print_output:
    print(raw_json)

  # Actually create the dictionary
  output_dict = json.loads(raw_json)
  return output_dict

# Create function that gets a column of a df as a comma separated string
def list_df_keys(df_col):
  output = ','.join(df_col.values.astype(str).tolist())
  return output

In [ ]:
def get_fantasy_games():
    """
    Retrieves the fantasy games associated wi\th the current logged-in user by making an API request to Yahoo Fantasy Sports.

    This function does not take any parameters as it assumes that the necessary authentication for the API request is handled elsewhere (e.g., set in headers or as environment variables).

    Returns:
    - A list of dictionaries, each representing a fantasy game with details such as 'game_key', 'game_sport' (the sport code), and 'game_season'.
    """
    games_dict = request_to_dict('https://fantasysports.yahooapis.com/fantasy/v2/users;use_login=1/games/teams')
    fantasy_games = games_dict['fantasy_content']['users']['user']['games']['game']
    games_list = [{'game_key': game['game_key'],
                   'game_sport': game['code'],
                   'game_season': game['season']
                   } for game in fantasy_games]
    return games_list


def get_leagues_for_game(selected_game_key):
    """
    Retrieves the leagues for a particular game based on the selected game key.

    Parameters:
    - selected_game_key: The key of the game for which to retrieve leagues.

    Returns:
    - A list of dictionaries, each containing information about a league within the game.
    """
    leagues_dict = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/users;use_login=1/games;game_keys={selected_game_key}/leagues')
    leagues_content = leagues_dict['fantasy_content']['users']['user']['games']['game']['leagues']
    if int(leagues_content['@count']) == 1:
        leagues = [leagues_content['league']]
    else:
        leagues = leagues_content['league']

    leagues_list = [{
        'league_key': league['league_key'],
        'league_id': league['league_id'],
        'league_name': league['name'],
        'scoring_type': league['scoring_type'],
        'start_week': league['start_week'],
        'start_date': league['start_date'],
        'end_date': league['end_date']
    } for league in leagues]

    return leagues_list

def get_teams_for_league(selected_league_key):
    """
    Retrieves the teams for a particular league based on the selected league key.

    Parameters:
    - selected_league_key: The key of the league for which to retrieve teams.

    Returns:
    - A list of dictionaries, each containing information about a team within the league.
    """
    teams_dict = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/league/{selected_league_key}/teams')
    teams_content = teams_dict['fantasy_content']['league']['teams']
    if int(teams_content['@count']) == 1:
        teams = [teams_content['team']]
    else:
        teams = teams_content['team']

    teams_list = [{
        'team_key': team['team_key'],
        'team_id': team['team_id'],
        'team_name': team['name'],
        'manager_name': team['managers']['manager']['nickname'],
        'manager_id': team['managers']['manager']['manager_id']
    } for team in teams]

    return teams_list

def print_and_select(options, prompt, print_keys):
    """
    Prints out a list of options and prompts the user to select one.

    Parameters:
    - options: A list of dictionaries containing the options to print.
    - prompt: A string to display as the input prompt.
    - print_keys: A list of keys to print from the options.

    Returns:
    - The selected option.
    """
    for option in options:
        print(': '.join(str(option[key]) for key in print_keys))
    return input(prompt)


def get_user_selection(game_key=False, league_key=False, team_key=False, opponent_key=False):
    """
    Handles the user interaction for selecting a game, league, and team.

    Parameters:
    - user_data: The dictionary containing the current logged-in user's information.

    Returns:
    - A dictionary with the selected game_key, league_key, and team_key.
    """

    global selection
    if 'selection' not in globals():
      selection = {}

    if 'game_key' not in selection and game_key:
      games_list = get_fantasy_games()
      selected_game_key = print_and_select(games_list, 'Please select the game_key that you want: ', ['game_sport', 'game_season', 'game_key'])
      selection['game_key'] = selected_game_key

    else:
      selected_game_key = selection['game_key']

    # When selecting a league
    if 'league_key' not in selection and league_key:
      leagues_list = get_leagues_for_game(selected_game_key)
      selected_league_key = print_and_select(leagues_list, 'Please select the league_key that you want: ', ['league_name', 'league_key'])
      selection['league_key'] = selected_league_key

    else:
      selected_league_key = selection['league_key']

    # When selecting a team
    if 'team_key' not in selection and team_key:
      teams_list = get_teams_for_league(selected_league_key)
      selected_team_key = print_and_select(teams_list, 'Please select the team_key that you want: ', ['team_name', 'team_key'])
      selection['team_key'] = selected_team_key

    # When selecting an opponent
    if 'opponent_key' not in selection and opponent_key:
      teams_list = get_teams_for_league(selected_league_key)
      selected_opponent_key = print_and_select(teams_list, 'Please select the team_key of your opponent: ', ['team_name', 'team_key'])
      selection['opponent_key'] = selected_opponent_key

    return selection

# Main code execution
selection = get_user_selection(game_key=True, league_key=True, team_key=True)
# selection = get_user_selection(league_key=True) #Figure out a way for it to determine which prerequisite keys are needed


In [ ]:
def check_selection(entity_key):
    """
    This function checks for the existence of the global dictionary 'selection'.
    If 'selection' exists, the function attempts to access the value with the
    provided entity_key within 'selection'. If the key does not exist, it calls
    the 'get_user_selection()' function to create or update the 'selection'
    dictionary.

    Args:
        entity_key: The key to be checked in the 'selection' dictionary.

    Returns:
        None.

    Raises:
        KeyError: If entity_key is not found in the 'selection' dictionary.
                  This is currently handled by the function by calling
                  'get_user_selection()'.

    Note:
        This function uses the global variable 'selection'. If 'selection' is not
        already defined, it will be created by calling 'get_user_selection()'.
    """
    global selection
    if 'selection' in globals():
        try:
            selection[entity_key]
            print('Checked that key exists in selection')
            return selection[entity_key]
        except KeyError:
            # In the future, can improve this function to only add the missing keys, rather than going through the whole flow again
            print('keyerror')
            kwargs = {entity_key: True}
            selection.update(get_user_selection(**kwargs))


    else:
        print('no var')
        selection = get_user_selection()


In [ ]:
check_selection('opponent_key')
# Not sure if this is needed for the script

In [ ]:
def get_league_stat_categories(league_key):
    """
    Retrieves the statistical categories for a given league.

    Parameters:
    - league_key: The key for the league whose stat categories are to be retrieved.

    Returns:
    - A list of dictionaries, each representing a stat category with details such as 'stat_id', 'stat_name', etc.
    """
    # Don't think I need this anymore -- passing the league key should mean that I don't need to check if it exists
    # check_selection(league_key)

    categories_dict = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/league/{league_key}/settings', print_output=True)

    stat_categories = categories_dict['fantasy_content']['league']['settings']['stat_categories']['stats']['stat']

    categories_list = [
        {
            'stat_id': stat['stat_id'],
            'stat_name': stat['name'],
            'stat_group': stat['group'],
            'abbreviation': stat['display_name'],
            'is_enabled': stat['stat_position_types']['stat_position_type'].get('is_only_display_stat', '0') != '1'
        }
        for stat in stat_categories
    ]

    return categories_list

league_stat_categories = get_league_stat_categories(selection['league_key']) ### Update this to use check_selection

# Get matchups for matchup analyser (dead end)

In [ ]:
# TODO: Think about how we might refactor this one

def get_matchups_for_team(team_key):
    """
    Retrieves the matchups for a particular team based on the team key.

    Parameters:
    - team_key: The key of the team for which to retrieve matchups.

    Returns:
    - A list of dictionaries, each containing information about a matchup.
    """
    matchups_dict = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/teams;team_keys={team_key}/matchups')
    matchup_data = matchups_dict['fantasy_content']['teams']['team']['matchups']['matchup']

    matchups_list = []
    for matchup in matchup_data:
        # Base matchup information
        matchup_info = {
            'week': matchup['week'],
            'week_start': matchup['week_start'],
            'week_end': matchup['week_end'],
            'teams': []
        }

        # Collect team and stats information
        for team in matchup['teams']['team']:
            team_info = {
                'team_key': team['team_key'],
                'team_id': team['team_id'],
                'team_name': team['name'],
                'stats': {stat['stat_id']: stat['value'] for stat in team['team_stats']['stats']['stat']}
            }
            matchup_info['teams'].append(team_info)

        matchups_list.append(matchup_info)

    return matchups_list


def select_and_display_matchups(matchups):
    """
    Displays the matchups and prompts the user to select one.

    Parameters:
    - matchups: A list of matchup information dictionaries.

    Returns:
    - The selected matchup dictionary.
    """
    # Display matchups
    for i, matchup in enumerate(matchups, start=1):
        teams_info = ' vs '.join([team['team_name'] for team in matchup['teams']])
        print(f"{i}: Week {matchup['week']} - {teams_info}")

    # Prompt for selection
    selected_index = int(input('Please select a matchup by number: ')) - 1
    return matchups[selected_index] if 0 <= selected_index < len(matchups) else None


team_key = selection['team_key'] ### Update this to use check_selection
matchups = get_matchups_for_team(team_key)
selected_matchup = select_and_display_matchups(matchups)


In [ ]:
def get_matchup_stats_reshaped(df):
    # ... (your existing code for fetching data into matchups_df)
    matchups_reshaped_list = []

    for i, row in df.iterrows():
        for team_counter in range(1, 3):  # assuming you have two teams in each matchup
            stats_dict = {
                'week': row['week'],
                'week_start': row['week_start'],
                'week_end': row['week_end'],
                'team': row[f'team_{team_counter}_name'],
            }

            for col in matchups_df.columns:
                if f'team_{team_counter}_' in col:
                    stat_name = col.replace(f'team_{team_counter}_', '')
                    if stat_name not in ['key', 'id', 'name']:
                        stats_dict['stat_name'] = stat_name
                        stats_dict['stat_value'] = row[col]
                        matchups_reshaped_list.append(stats_dict.copy())

    matchups_reshaped_df = pd.DataFrame(matchups_reshaped_list)
    matchups_pivoted_df = matchups_reshaped_df.pivot(index='stat_name', columns='team', values='stat_value')

    return matchups_pivoted_df

matchups_pivoted_df = get_matchup_stats_reshaped(week_matchup_df)

# Get rosters and player stats per roster (dead end)

In [ ]:
# Function to get stats for a given player. Right now just set to season, but can split out a url constructor for different options in the future
def get_stats(player_key, season=2023):
  player_row = {}
  stat_data = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/player/{player_key}/stats;type=season;season={season}', print_output=True)
  time.sleep(0.25)

  # Consider calling the function here in case the var is not already defined
  # categories_lookup_df = pd.DataFrame(league_stat_categories) #This is removed for now since we're just calling the var directxly

  stats = stat_data['fantasy_content']['player']['player_stats']['stats']['stat']

  for stat_category in stats:
    id = stat_category['stat_id']
    # one of these stats are games played, but not sure
    if id == '0':
      player_row['games_played'] = stat_category['value']

    for stat in league_stat_categories:
      if stat['stat_id'] == id:
        stat_name = stat['stat_name']

        # Replace - with 0s
        if stat_category['value'] == '-':
          player_row[stat_name] = 0

        else:
          player_row[stat_name] = stat_category['value']

        # Calculate per game values, except for GA and GAA
        if (id not in ['22', '23']) & (stat_category['value'] != '-') & (player_row['games_played'] != '-'):
          player_row[f"{stat_name}_pg"] = float(stat_category['value']) / float(player_row['games_played'])

        elif stat_category['value'] == '-':
          player_row[f"{stat_name}_pg"] = 0

        else:
          player_row[f"{stat_name}_pg"] = float(stat_category['value'])

  return player_row

# Get list of players for a given team
import time
def get_roster(team_key, stats=False, format='df'):
    player_data = request_to_dict(f"https://fantasysports.yahooapis.com/fantasy/v2/team/{team_key}/roster")

    # Retrieve details about each player on the roster
    players = player_data['fantasy_content']['team']['roster']['players']['player']
    df = pd.DataFrame()
    rows = []
    for player in players:
      player_row = {
        'player_key':player['player_key']
        , 'player_name':player['name']['full']
        , 'player_position':player['display_position']
        , 'eligible_positions':player['eligible_positions']['position']
        , 'is_undroppable':player['is_undroppable']
        , 'team_key':player['editorial_team_key']
        , 'team_abbreviation':player['editorial_team_abbr']
        , 'team_name':player['editorial_team_full_name']
        , 'fantasy_team_key':player_data['fantasy_content']['team']['team_key']
        , 'fantasy_team_name':player_data['fantasy_content']['team']['name']
      }

      try:
        player_row['satus'] = player['status']

      except KeyError:
        pass

      if stats:
        stats = get_stats(player['player_key'])
        print(stats)
        for key, value in stats.items():
          player_row[key] = value

      print(player_row)
      rows.append(player_row)

    data = pd.DataFrame(rows)
    df = pd.concat([df, data], ignore_index=True)

    return df

In [ ]:
# This gets all the fantasy rosters across the league. Next thing to get stats per player
league_rosters = pd.DataFrame()

league_teams = get_teams_for_league(check_selection('league_key'))
for team in league_teams:
  df = get_roster(team['team_key'], stats=True)
  league_rosters = pd.concat([league_rosters, df], ignore_index=True)


In [ ]:
league_rosters

# Calculating expected stats (dead end)
Come back to explain what happens

In [ ]:
# Retrieve the rosters of the user's team, and that of the selected matchup
user_roster_df = get_roster(check_selection('team_key'))
matchup_opponent_roster_df = get_roster(check_selection('opponent_key'))

In [ ]:
def get_nhl_games(season):
    base_url = "https://statsapi.web.nhl.com/api/v1/schedule"
    params = {
        "season": season
    }

    response = requests.get(base_url, params=params)

    if response.status_code != 200:
        print(f"Error: Unable to fetch data. Status code: {response.status_code}")
        return None

    data = response.json()
    games_list = []

    for date in data['dates']:
        for game in date['games']:
            game_info = {
                "game_id": game['gamePk'],
                "season": game['season'],
                "date": date['date'],
                "home_team": game['teams']['home']['team']['name'],
                "away_team": game['teams']['away']['team']['name'],
                "home_score": game['teams']['home'].get('score', 'N/A'),
                "away_score": game['teams']['away'].get('score', 'N/A')
            }
            games_list.concat(game_info)

    # print(data)

    df = pd.DataFrame(games_list)
    return df

nhl_schedule = get_nhl_games('20232024')

In [ ]:
def get_team_game_stats(df, user_date):
    # Convert 'date' column to datetime format
    df['date'] = pd.to_datetime(df['date'])

    # Convert user_date to datetime format
    user_date = datetime.strptime(user_date, '%Y-%m-%d')

    # Find the Monday of the week of user_date
    monday_of_week = user_date - timedelta(days=user_date.weekday())

    # Find the Sunday of the week of user_date
    sunday_of_week = monday_of_week + timedelta(days=6)

    # Filter games scheduled for that week
    df_week = df[(df['date'] >= monday_of_week) & (df['date'] <= sunday_of_week)]

    # Create a list of all teams playing that week
    all_teams = list(set(df_week['home_team']).union(set(df_week['away_team'])))

    # Initialize empty lists to store results
    teams = []
    total_games = []
    remaining_games = []

    # Loop through each team to calculate stats
    for team in all_teams:
        # Count total games for this team for the week
        total = len(df_week[(df_week['home_team'] == team) | (df_week['away_team'] == team)])

        # Count remaining games for this team for the week on or after user_date
        remaining = len(df_week[((df_week['home_team'] == team) | (df_week['away_team'] == team)) & (df_week['date'] >= user_date)])

        # Append to lists
        teams.append(team)
        total_games.append(total)
        remaining_games.append(remaining)

    # Create a DataFrame to store the results
    result_df = pd.DataFrame({
        'team': teams,
        'total_games': total_games,
        'remaining_games': remaining_games
    })

    return result_df

In [ ]:
user_date = '2023-11-03'
remaining_games = get_team_game_stats(nhl_schedule, user_date)
remaining_games

In [ ]:
def calculate_expected_remaining_stats(roster_df, team_games_remaining_df):
    # Create a DataFrame to hold the expected remaining stats
    expected_stats_df = pd.DataFrame()

    # Iterate through each player in the roster DataFrame
    for _, player_row in roster_df.iterrows():
        expected_stats_row = {
            'player_key': player_row['player_key'],
            'player_name': player_row['player_name'],
        }

        # Retrieve the number of games remaining for the player's team this week
        team_key = player_row['team_name']
        games_remaining = team_games_remaining_df.loc[team_games_remaining_df['team'] == team_key, 'remaining_games'].values[0]

        # Calculate the expected remaining stats based on per-game stats
        for column in roster_df.columns:
            if "_pg" in column:
                stat_name = column.replace("_pg", "")

                if column == "Goals Against Average_pg":
                  expected_value = player_row[column]

                else:
                  expected_value = player_row[column] * games_remaining

                expected_stats_row[f"expected_{stat_name}"] = expected_value

        expected_stats_df = expected_stats_df.append(expected_stats_row, ignore_index=True)

        # Sum all numeric columns except 'expected_Goals Against Average'
        sum_values = expected_stats_df.drop(columns=['expected_Goals Against Average']).sum(numeric_only=True)

        # Calculate the average for 'expected_Goals Against Average'
        average_gaa = expected_stats_df['expected_Goals Against Average'].mean()

        # Combine the two into a single Series
        total_values = round(sum_values.append(pd.Series({'expected_Goals Against Average': average_gaa})))

        # Optionally, convert to DataFrame for better presentation
        total_values_df = total_values.to_frame().reset_index()
        total_values_df.columns = ['Stat', 'Total Value']

    # return expected_stats_df
    return total_values_df, expected_stats_df

user_expected_stats_df, user_expected_stats_audit = calculate_expected_remaining_stats(user_roster_df, remaining_games)
opponent_expected_stats_df, opp_expected_stats_audit = calculate_expected_remaining_stats(matchup_opponent_roster_df, remaining_games)

In [ ]:
user_expected_stats_df

In [ ]:
def add_expected_stats_to_matchup(matchups_pivoted_df, user_expected_stats_df, user_team_name, opponent_expected_stats_df, opponent_team_name):
    # Create a copy of the original DataFrame
    updated_df = matchups_pivoted_df.copy()

    # Function to update stats for a single team
    def update_team_stats(expected_stats_df, team_name):
        for _, row in expected_stats_df.iterrows():
            stat_name = row['Stat'].replace('expected_', '')  # Assuming the stat names in expected_stats_df start with 'expected_'
            expected_value = float(row['Total Value'])  # Convert to float

            # Check if the stat exists in matchups_pivoted_df
            if stat_name in updated_df.index:
                current_value = updated_df.loc[stat_name, team_name]

                # Handle NaNs and convert to float
                current_value = 0 if pd.isna(current_value) else float(current_value)

                # Add the expected value to the current value
                updated_df.loc[stat_name, team_name] = current_value + expected_value

    # Update stats for user and opponent
    update_team_stats(user_expected_stats_df, user_team_name)
    update_team_stats(opponent_expected_stats_df, opponent_team_name)

    return updated_df


updated_df = add_expected_stats_to_matchup(matchups_pivoted_df, user_expected_stats_df, 'AHO-V-O Crew', opponent_expected_stats_df, 'Beelievers')
updated_df

# Getting historical matchup data here

In [ ]:
def get_weekly_stats_for_team(team_key, week, stat_categories):
    """
    Retrieves the weekly stats for a specific team in a league, with stat names instead of stat IDs.

    Parameters:
    - team_key: The key of the team.
    - week: The specific week number for which stats are to be retrieved.
    - stat_categories: A list of stat categories with names and IDs, obtained from get_league_stat_categories().

    Returns:
    - A dictionary containing the team's stats for the specified week with stat names as keys.
    """
    # Create a mapping of stat IDs to names using the provided stat_categories list
    id_to_name = {stat['stat_id']: stat['stat_name'] for stat in stat_categories if stat['is_enabled']}

    # Perform the API request to get weekly stats for the given team and week
    stats_dict = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/team/{team_key}/stats;type=week;week={week}')

    # Extract the stats from the response dictionary
    if 'team' in stats_dict['fantasy_content']:
        team_stats_data = stats_dict['fantasy_content']['team']['team_stats']['stats']
        # Initialize a dictionary for week stats with stat names
        week_stats = {'week': week, 'stats': {}}
        for stat in team_stats_data['stat']:
            # Use the stat ID to find the stat name and map the value
            stat_name = id_to_name.get(stat['stat_id'], f"Unknown Stat ID: {stat['stat_id']}")
            week_stats['stats'][stat_name] = stat['value']

        return week_stats
    else:
        # Handle the case where 'team' might not be in the response
        return {'week': week, 'stats': {}, 'error': 'No data available for this week.'}

# Example usage:
# Assume team_key is available from the user's selections, and stat_categories is fetched from get_league_stat_categories()
league_stat_categories = get_league_stat_categories(selection['league_key'])
# team_weekly_stats = get_weekly_stats_for_team(selection['team_key'], 5, league_stat_categories)


In [ ]:
def get_league_weekly_stats(league_key, week_ending, week_starting=1):
    """
    Retrieves weekly stats for all teams in a given league up to a specified week.

    This function collects the statistics for each team in the league on a week-by-week basis,
    aggregating them into a list of dictionaries, each containing a team's weekly stats.

    Parameters:
    - league_key: The unique identifier for the league.
    - week_ending: The last week number (inclusive) for which stats should be retrieved.
    - week_starting: The first week number for which stats should be retrieved. Defaults to 1

    Returns:
    - A list of dictionaries where each dictionary contains the name of a team and its stats for each week
      from week 1 up to and including week_ending.
    """

    # It's more efficient to make single API requests outside the loop if they don't depend on the iteration
    team_list = get_teams_for_league(league_key)
    league_stat_categories = get_league_stat_categories(league_key)

    output = []
    for team in team_list:
        for week in range(week_starting, week_ending+1):
            # get_weekly_stats_for_team could fail, consider try-except or check the return value
            team_stats = get_weekly_stats_for_team(team['team_key'], week, league_stat_categories)
            # Ensure that 'team' key in team_stats dict does not overwrite any existing keys
            team_stats['team_name'] = team['team_name']
            output.append(team_stats)

    return output

# TODO need to update this so that we only get enabled stats

In [ ]:
categories_performance = get_league_weekly_stats(selection['league_key'], 17, week_starting=8)

In [ ]:
import pandas as pd

def convert_to_dataframe(weekly_stats):
    """
    Converts a list of weekly stats for teams into a pandas DataFrame.

    Each row in the DataFrame corresponds to a single team's stats for a particular week.

    Parameters:
    - weekly_stats: A list of dictionaries containing team names and their stats per week.

    Returns:
    - A pandas DataFrame with one row per team per week and stats as columns.
    """

    # Flatten the list of dictionaries into a list of rows, expanding the stats dictionary
    rows = []
    for weekly_stat in weekly_stats:
        row = {'team_name': weekly_stat['team_name'], 'week': weekly_stat['week']}
        row.update(weekly_stat['stats'])  # Adds all stat categories as separate columns
        rows.append(row)

    # Create the DataFrame
    df = pd.DataFrame(rows)

    return df

# Example usage:
# Assume league_weekly_stats is the output from get_league_weekly_stats function
df = convert_to_dataframe(categories_performance)


In [ ]:
df = df.reset_index(drop=True)
df = df.drop(columns=['Unknown Stat ID: 22'])

In [ ]:
df = df.set_index('team_name', drop=True)
df

# Player Roster & Stats Testing Grounds (dead end)

In [ ]:
# Testing to see if I can get stats for a specific week
user_roster_df = get_roster(check_selection('team_key'))

In [ ]:
user_roster_df

In [ ]:
player_key = '427.p.6777'
date_start = '2024-11-03'
date_end = '2024-11-10'
base_url = f'https://fantasysports.yahooapis.com/fantasy/v2/player/{player_key}/stats;type=week;week={date_start}'
date_only_url = f'https://fantasysports.yahooapis.com/fantasy/v2/player/{player_key}/stats;date={date_start}'

test_stats = request_to_dict(date_only_url, print_output = True)

date_end_only_url = f'https://fantasysports.yahooapis.com/fantasy/v2/player/{player_key}/stats;date={date_end}'
test_stats = request_to_dict(base_url, print_output = True)

In [ ]:
test_stats

In [ ]:
# Looks like it's possible to get a specific date's stats. Good alternative to pull, though going to be a bit of a pain in the ass to get a lot of data
week_test = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/player/{player_key}/stats;type=date;date=2024-02-19')


# Sweet, can get season stats for any year. This is good.
# Next challenge is getting a list of all the players that we can loop through to get these stats if we want to create the draft helper
season_test = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/player/{player_key}/stats;type=season;season=2023')
# season_test

or_test = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/player/{player_key}/stats;sort=60')
or_test

In [ ]:
# This gives us a list of players in the league. Need to figure out pagination, but this is a good start
league_key = check_selection('league_key')
all_player_test = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/leagues;league_keys={league_key}/players;sort=OR;season=2023;start=24;out=stats') #This sorts by overall rank for 2023, starting from 24th record
# I'm pretty sure the counting is 0 based
# From here we can just write a function that keeps going until we get an error
all_player_test

In [ ]:
import time

def extract_player_stats(player_data, cumulative_stats):
    for player in player_data['fantasy_content']['leagues']['league']['players']['player']:
        player_id = player["player_id"]
        player_name = player["name"]["full"]

        stats = {
            "player_name": player_name,
        }

        # Extract basic stats
        if "player_stats" in player:
            for stat in player["player_stats"]["stats"]["stat"]:
                stats[f"stat_{stat['stat_id']}"] = stat["value"]

        # Extract advanced stats
        if "player_advanced_stats" in player:
            for stat in player["player_advanced_stats"]["stats"]["stat"]:
                stats[f"advanced_stat_{stat['stat_id']}"] = stat["value"]

        cumulative_stats[player_id] = stats

cumulative_stats = {}
n = 0
while n < 500:
    print(f'starting from {n}')
    data = request_to_dict(f'https://fantasysports.yahooapis.com/fantasy/v2/leagues;league_keys={league_key}/players;sort=OR;season=2023;start={n};out=stats')
    extract_player_stats(data, cumulative_stats)
    time.sleep(1)
    print(f'completed {n}, dict size = {len(cumulative_stats)}')
    n += 25  # Assuming 25 players are returned per request

# Convert the cumulative dictionary to a DataFrame
df = pd.DataFrame.from_dict(cumulative_stats, orient='index')

# Display the DataFrame
print(df)



In [ ]:
df = pd.DataFrame.from_dict(cumulative_stats, orient='index')
df

In [ ]:
# This gets all the fantasy rosters across the league. Next thing to get stats per player
league_rosters = pd.DataFrame()

league_teams = get_teams_for_league(check_selection('league_key'))
for team in league_teams:
  df = get_roster(team['team_key'], stats=True)
  league_rosters = pd.concat([league_rosters, df], ignore_index=True)
